# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Samra-ca/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one anonymized content page snapshot. The starter CSV is a trailing performance snapshot for a page, not a calendar-month row.

I am using the page-level snapshot from `data/raw/content_refresh_anonymized.csv` with:
- `impressions_90d`, `clicks_90d`, `sessions_90d` for the current 90-day performance window
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` for the most recent 30 days
- `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` for the prior 30 days

This contract is built on the page snapshot slice. I deliberately exclude label-derived fields such as `trend_direction` and `trend_pct` from the feature set.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature:
- `avg_position`: knowable before the decision because it is the current search ranking position for the page.
- `ctr`: knowable before the decision because it is derived from current clicks and impressions in the snapshot.
- `word_count`: knowable before the decision because it is a static page content attribute.
- `content_age_days`: knowable before the decision because page age is already known.
- `days_since_last_update`: knowable before the decision because the last update timestamp is available.

Label / proxy:
- `is_declining_label = trend_direction == 'down'`: the target for movement risk.
- `trend_direction`: label source, not a feature.
- `trend_pct`: label source rule, not a feature.

Context:
- `content_id`, `client_id`: grouping / splitting keys only.
- `content_type`, `main_intent`: segment metadata for analysis, not core model features.

Excluded:
- `trend_direction`, `trend_pct`: derived from the same 60-day window as the label and therefore leak.
- `provider_used`, `model_used`: sparse/delivery metadata, not stable prediction signals.
- `content_id`, `client_id`: pseudonymous IDs only for grouping and split validation.


In [1]:
# Load the starter snapshot and show its shape
import pandas as pd

path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(path)
print('rows, cols:', df.shape)
print('unique content_id:', df['content_id'].nunique())
print('duplicate content_id rows:', df.shape[0] - df['content_id'].nunique())
print('\nTrend direction counts:')
print(df['trend_direction'].value_counts())
print('\nTrend pct present:', 'trend_pct' in df.columns)

chosen_features = ['avg_position', 'ctr', 'word_count', 'content_age_days', 'days_since_last_update']
print('\nChosen features and missingness:')
for col in chosen_features:
    print(f' - {col}: missing={df[col].isna().sum()}, unique={df[col].nunique()}')

print('\nDeclining label fraction:', df['trend_direction'].eq('down').mean())
print('avg_position == 0 rows:', (df['avg_position'] == 0).sum(), '(0 means no position data)')

df['is_declining_label'] = df['trend_direction'].eq('down')
print('\nLabel-derived columns that will be excluded:')
print(df[['trend_direction', 'trend_pct']].head(5).to_string(index=False))


rows, cols: (30000, 44)
unique content_id: 30000
duplicate content_id rows: 0

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend pct present: True

Chosen features and missingness:
 - avg_position: missing=0, unique=869
 - ctr: missing=0, unique=401
 - word_count: missing=7699, unique=5476
 - content_age_days: missing=0, unique=225
 - days_since_last_update: missing=0, unique=57

Declining label fraction: 0.5420666666666667
avg_position == 0 rows: 1205 (0 means no position data)

Label-derived columns that will be excluded:
trend_direction  trend_pct
           down      -41.4
           down      -57.7
           down      -60.9
         stable      -13.8
           down      -34.7


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


This section shows the row grain, counts, availability checks, the five-feature frame, and the deliberate leakage experiment.

In [2]:
# Verify contract claims with data checks
print('1) Grain check: one row per content_id')
grain_dup = df.duplicated(subset=['content_id']).sum()
print('duplicate content_id rows:', grain_dup)

print('\n2) Row count and snapshot coverage')
print('total rows:', len(df))
print('rows with impressions_last_30d > 0:', (df['impressions_last_30d'] > 0).sum())
print('rows with impressions_prev_30d >= 0:', (df['impressions_prev_30d'] >= 0).sum())
print('rows with sessions_90d > 0:', (df['sessions_90d'] > 0).sum())

print('\n3) Availability checks using boolean filtering')
valid_pos = df['avg_position'] > 0
print('rows with valid avg_position (>0):', valid_pos.sum(), f'({valid_pos.mean():.1%})')
valid_ctr = df['ctr'].notna()
print('rows with valid ctr:', valid_ctr.sum(), f'({valid_ctr.mean():.1%})')
valid_sessions = df['sessions_90d'] > 0
print('rows with sessions_90d > 0:', valid_sessions.sum(), f'({valid_sessions.mean():.1%})')

# Build a small feature frame with five selected features
feature_frame = df.loc[:, ['content_id', 'client_id', 'avg_position', 'ctr', 'word_count', 'content_age_days', 'days_since_last_update']].copy()
print('\nFeature frame preview:')
print(feature_frame.head(5).to_string(index=False))
print('\nFeature frame shape:', feature_frame.shape)
print('Feature columns:', feature_frame.columns.tolist()[2:])

# Feature availability lines
feature_availability = {
    'avg_position': 'Knowable at decision moment because current search position is part of the snapshot.',
    'ctr': 'Knowable at decision moment because current clicks and impressions are already recorded.',
    'word_count': 'Knowable at decision moment because page length exists before the prediction.',
    'content_age_days': 'Knowable at decision moment because content age is already known.',
    'days_since_last_update': 'Knowable at decision moment because the last update time is already recorded.'
}
print('\nFeature availability:')
for f, reason in feature_availability.items():
    print(f'- {f}: {reason}')

# Deliberate leakage experiment
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

safe_features = ['avg_position', 'ctr', 'word_count', 'content_age_days', 'days_since_last_update']
X_safe = feature_frame[safe_features].fillna(-1)
y = df['trend_direction'].eq('down').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X_safe, y, stratify=y, random_state=42, test_size=0.3)
model_safe = LogisticRegression(max_iter=1000, solver='liblinear')
model_safe.fit(X_train, y_train)
y_pred_safe = model_safe.predict(X_test)
safe_acc = accuracy_score(y_test, y_pred_safe)
safe_auc = roc_auc_score(y_test, model_safe.predict_proba(X_test)[:, 1])
print(f'\nHonest model (no label-derived features): accuracy={safe_acc:.3f}, AUC={safe_auc:.3f}')

X_leak = df[safe_features + ['trend_pct']].fillna(-1)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, stratify=y, random_state=42, test_size=0.3)
model_leak = LogisticRegression(max_iter=1000, solver='liblinear')
model_leak.fit(X_train_l, y_train_l)
y_pred_leak = model_leak.predict(X_test_l)
leak_acc = accuracy_score(y_test_l, y_pred_leak)
leak_auc = roc_auc_score(y_test_l, model_leak.predict_proba(X_test_l)[:, 1])
print(f'Leaky model with label-derived trend_pct: accuracy={leak_acc:.3f}, AUC={leak_auc:.3f}')
print('Leakage lesson: adding trend_pct makes the score much higher, but it is not allowed as a feature.')


1) Grain check: one row per content_id
duplicate content_id rows: 0

2) Row count and snapshot coverage
total rows: 30000
rows with impressions_last_30d > 0: 27453
rows with impressions_prev_30d >= 0: 30000
rows with sessions_90d > 0: 30000

3) Availability checks using boolean filtering
rows with valid avg_position (>0): 28795 (96.0%)
rows with valid ctr: 30000 (100.0%)
rows with sessions_90d > 0: 30000 (100.0%)

Feature frame preview:
          content_id         client_id  avg_position  ctr  word_count  content_age_days  days_since_last_update
content_304f48230142 client_f369cb89fc          10.6 0.76      3221.0               187                      20
content_a1fb4e703a9e client_4e07408562          20.3 0.05      2481.0               445                      25
content_9aa793d4d895 client_7f2253d7e2          36.5 0.09      3515.0               141                      20
content_331d6c4de07b client_19581e27de           6.2 0.49         NaN               463                      22

## 4. Data limits

This data can never tell me whether a page update caused the traffic change. It only shows observed search performance and engagement in the snapshot window.

Limitations:
- There is no true future window in the starter CSV. `trend_direction` is derived from the same page's last 60 days, so it is not a causal future label.
- `avg_position = 0` means no ranking data, not a literal position of 0.
- `provider_used` and `model_used` are sparse/mostly missing, so they are weak for a general model.
- Pages with zero `impressions_prev_30d` get `trend_pct` set to 0 by prep; that is a derived rule, not a signal.


In [3]:
print('This notebook uses the starter snapshot in data/raw/content_refresh_anonymized.csv.')
print('The dataset does not contain an explicit report month column, so the contract is based on the page snapshot window.')
print('avg_position == 0 means no ranking data, not a real position. trend_pct / trend_direction are label-derived rules.')


This notebook uses the starter snapshot in data/raw/content_refresh_anonymized.csv.
The dataset does not contain an explicit report month column, so the contract is based on the page snapshot window.
avg_position == 0 means no ranking data, not a real position. trend_pct / trend_direction are label-derived rules.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.